In [ ]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# Load prepared training data
X_train = pd.read_csv("../data/processed/X_train.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").iloc[:, 0]

X = X_train
y = y_train

print("Training data shape:", X.shape)
print("Churn rate in training set: {:.2%}".format(y.mean()))

In [ ]:
# Use prepared training data for cross-validation
X_train = pd.read_csv("../data/processed/X_train.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").values.ravel()

print("Train shape:", X_train.shape)

Dataset shape: (4238, 25)


,Customer ID,Recency,Frequency,Monetary,AvgOrderValue,UniqueProducts,TotalItems,AvgDaysBetweenPurchases,BasketSize,PreferredDay,...,Purchases_Last60Days,Purchases_Last90Days,ProductDiversityScore,AvgPricePreference,StdPricePreference,R_Score,F_Score,M_Score,RFM_Score,CustomerSegment
0,12347.0,2,7,3643.58,21.061156,99,1924,2.104651,274.857143,Tuesday,...,2.0,2.0,99,2.710867,2.283755,4,4,4,444,Champions
1,12348.0,75,4,450.20,45.020000,7,149,31.333333,37.250000,Thursday,...,0.0,1.0,7,16.390000,20.320581,2,3,2,232,Regular
2,12349.0,19,1,1667.55,23.486620,71,559,0.000000,559.000000,Monday,...,1.0,1.0,71,8.487324,35.504405,3,1,4,314,At Risk
3,12350.0,310,1,334.40,19.670588,17,197,0.000000,197.000000,Wednesday,...,0.0,0.0,17,3.841176,9.334751,1,1,2,112,Regular
4,12352.0,36,8,2506.04,29.482824,59,536,3.071429,67.000000,Tuesday,...,1.0,3.0,59,15.930706,53.706324,3,4,4,344,Loyal


In [ ]:
# Prepare data and model for CV
X = X_train
y = y_train

print("Churn distribution in training set:")
print(pd.Series(y).value_counts(normalize=True))

Churn distribution:
Churn
0    2829
1    1409
Name: count, dtype: int64


In [4]:
leakage_columns = [
    "CustomerID",
    "Churn",
    "Recency",
    "Frequency",
    "Monetary",
    "TotalOrders",
    "AvgOrderValue",
    "PurchaseVelocity",
    "Purchases_Last30Days",
    "Purchases_Last60Days",
    "Purchases_Last90Days"
]

X = df.drop(columns=leakage_columns, errors="ignore")
y = df["Churn"]

categorical_cols = X.select_dtypes(include=["object"]).columns

for col in categorical_cols:
    X[col] = X[col].astype("category").cat.codes

print("Feature shape:", X.shape)
print("Categorical columns:", list(categorical_cols))

Feature shape: (4238, 17)
Categorical columns: ['PreferredDay', 'CustomerSegment']


In [ ]:
# Best model: Logistic Regression
model = LogisticRegression(max_iter=1000)
print("Logistic Regression model created for CV")

Pipeline created


In [ ]:
kfold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Stratified 5-Fold initialized")

Stratified 5-Fold initialized


In [ ]:
model = LogisticRegression(max_iter=1000)

fold = 1
roc_auc_scores = []

for train_index, val_index in kfold.split(X, y):
    print(f"\nFold {fold}")
    X_train_cv, X_val_cv = X.iloc[train_index], X.iloc[val_index]
    y_train_cv, y_val_cv = y.iloc[train_index], y.iloc[val_index]

    model.fit(X_train_cv, y_train_cv)
    y_val_prob = model.predict_proba(X_val_cv)[:, 1]

    fold_roc_auc = roc_auc_score(y_val_cv, y_val_prob)
    roc_auc_scores.append(fold_roc_auc)
    print(f"ROC-AUC (validation): {fold_roc_auc:.4f}")

    fold += 1

print("\nCross-validation completed.")
print(f"Mean ROC-AUC: {np.mean(roc_auc_scores):.4f}")
print(f"Std ROC-AUC: {np.std(roc_auc_scores):.4f}")

Fold: 1
ROC-AUC: 0.976
Precision: 1.0
Recall: 0.7305
------------------------
Fold: 2
ROC-AUC: 0.9752
Precision: 0.9864
Recall: 0.7695
------------------------
Fold: 3
ROC-AUC: 0.9738
Precision: 1.0
Recall: 0.7553
------------------------
Fold: 4
ROC-AUC: 0.977
Precision: 1.0
Recall: 0.7438
------------------------
Fold: 5
ROC-AUC: 0.9755
Precision: 1.0
Recall: 0.766
------------------------


In [ ]:
import matplotlib.pyplot as plt

fold_indices = np.arange(1, len(roc_auc_scores) + 1)

plt.figure(figsize=(8, 5))
plt.plot(fold_indices, roc_auc_scores, marker="o", label="CV ROC-AUC per fold")
plt.axhline(y=np.mean(roc_auc_scores), color="green", linestyle="--", label="Mean CV ROC-AUC")
plt.title("5-Fold Cross-Validation ROC-AUC (Logistic Regression)")
plt.xlabel("Fold")
plt.ylabel("ROC-AUC")
plt.ylim(0.5, 1.0)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

Average Cross Validation Results
ROC-AUC: 0.9755
Precision: 0.9973
Recall: 0.753


In [9]:
results = pd.DataFrame({
    "Fold":[1,2,3,4,5],
    "ROC_AUC":roc_scores,
    "Precision":precision_scores,
    "Recall":recall_scores
})

results

,Fold,ROC_AUC,Precision,Recall
0,1,0.976042,1.000000,0.730496
1,2,0.975215,0.986364,0.769504
2,3,0.973849,1.000000,0.755319
3,4,0.977019,1.000000,0.743772
4,5,0.975529,1.000000,0.765957
